# 10

In [38]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split

covtype = fetch_covtype()

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    covtype.data, covtype.target, random_state=42
)

In [43]:
X_train.data.shape

(435759, 54)

In [28]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

mlp_clf = MLPClassifier(hidden_layer_sizes=[100, 100, 100], early_stopping=True,
                        verbose=True, random_state=42)
pipeline = make_pipeline(StandardScaler(), mlp_clf)
pipeline.fit(X_train, y_train)

Iteration 1, loss = 0.56620846
Validation score: 0.797870
Iteration 2, loss = 0.43539499
Validation score: 0.828323
Iteration 3, loss = 0.38018900
Validation score: 0.853314
Iteration 4, loss = 0.34512917
Validation score: 0.863296
Iteration 5, loss = 0.31965615
Validation score: 0.872636
Iteration 6, loss = 0.30171112
Validation score: 0.877295
Iteration 7, loss = 0.28662288
Validation score: 0.883307
Iteration 8, loss = 0.27596983
Validation score: 0.888517
Iteration 9, loss = 0.26411366
Validation score: 0.890146
Iteration 10, loss = 0.25557785
Validation score: 0.893083
Iteration 11, loss = 0.24815619
Validation score: 0.898316
Iteration 12, loss = 0.24179882
Validation score: 0.898637
Iteration 13, loss = 0.23527023
Validation score: 0.901551
Iteration 14, loss = 0.22882145
Validation score: 0.901666
Iteration 15, loss = 0.22431561
Validation score: 0.905980
Iteration 16, loss = 0.22056695
Validation score: 0.906715
Iteration 17, loss = 0.21544139
Validation score: 0.901437
Iterat

,steps,"[('standardscaler', ...), ('mlpclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,hidden_layer_sizes,"[100, 100, ...]"
,activation,'relu'
,solver,'adam'
,alpha,0.0001


In [41]:
mlp_clf.best_validation_score_

0.9300302919038003

In [42]:
pipeline.score(X_test, y_test)

0.928841400866075

We tested with hidden layers (100, 100, 100) and it could achieve 92.88% accuracy on test set (not >93%). We need to do hyperparameter tuning.

In [49]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from scipy.stats import loguniform

pipeline_new = make_pipeline(
    StandardScaler(), 
    MLPClassifier(early_stopping=True, verbose=True, random_state=42))

param_dist = {
    "mlpclassifier__hidden_layer_sizes": [(100,), (100, 100), (100, 50), (100, 100, 100), (100, 50, 25)],
    "mlpclassifier__alpha": uniform(0.0001, 0.01),
    "mlpclassifier__learning_rate_init": loguniform(0.001, 0.1),
    "mlpclassifier__activation": ["relu", "tanh"],
}

search = RandomizedSearchCV(
    pipeline_new,
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=1
)

X_small, _, y_small, _ = train_test_split(
    X_train,
    y_train,
    train_size=100000,
    stratify=y_train,
    random_state=42
)

search.fit(X_small, y_small)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Iteration 1, loss = 0.67700512
Validation score: 0.735400
Iteration 2, loss = 0.57323285
Validation score: 0.760700
Iteration 3, loss = 0.52811148
Validation score: 0.780000
Iteration 4, loss = 0.49019128
Validation score: 0.790600
Iteration 5, loss = 0.45936904
Validation score: 0.804400
Iteration 6, loss = 0.43258967
Validation score: 0.812600
Iteration 7, loss = 0.41002343
Validation score: 0.830200
Iteration 8, loss = 0.38995737
Validation score: 0.835400
Iteration 9, loss = 0.37204049
Validation score: 0.844800
Iteration 10, loss = 0.35668522
Validation score: 0.849500
Iteration 11, loss = 0.34466392
Validation score: 0.856100
Iteration 12, loss = 0.33174431
Validation score: 0.862700
Iteration 13, loss = 0.32125546
Validation score: 0.866300
Iteration 14, loss = 0.31042196
Validation score: 0.866700
Iteration 15, loss = 0.30192936
Validation score: 0.873900
Iteration 16, loss = 0.29490172
Validation score: 0.872900
Iter

d:\programming\machine_learning\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


,estimator,Pipeline(step...rbose=True))])
,param_distributions,"{'mlpclassifier__activation': ['relu', 'tanh'], 'mlpclassifier__alpha': <scipy.stats....001454B6902C0>, 'mlpclassifier__hidden_layer_sizes': [(100,), (100, ...), ...], 'mlpclassifier__learning_rate_init': <scipy.stats....001454B5FB110>}"
,n_iter,30
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [53]:
import pandas as pd

results = pd.DataFrame(search.cv_results_)
top5 = results.sort_values(by="mean_test_score", ascending=False).head(10)

print(top5[["mean_test_score", "params"]])

    mean_test_score                                             params
16          0.90169  {'mlpclassifier__activation': 'tanh', 'mlpclas...
14          0.89854  {'mlpclassifier__activation': 'tanh', 'mlpclas...
4           0.89478  {'mlpclassifier__activation': 'tanh', 'mlpclas...
2           0.89020  {'mlpclassifier__activation': 'relu', 'mlpclas...
29          0.88615  {'mlpclassifier__activation': 'relu', 'mlpclas...
13          0.88400  {'mlpclassifier__activation': 'tanh', 'mlpclas...
10          0.88324  {'mlpclassifier__activation': 'tanh', 'mlpclas...
21          0.87788  {'mlpclassifier__activation': 'tanh', 'mlpclas...
17          0.87608  {'mlpclassifier__activation': 'tanh', 'mlpclas...
24          0.87235  {'mlpclassifier__activation': 'relu', 'mlpclas...


In [54]:
search.best_params_

{'mlpclassifier__activation': 'tanh',
 'mlpclassifier__alpha': np.float64(0.009366588657937942),
 'mlpclassifier__hidden_layer_sizes': (100, 100, 100),
 'mlpclassifier__learning_rate_init': np.float64(0.0012315571723666018)}